In [ ]:
import os
from typing import Literal
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.messages import HumanMessage
from rich import print as rprint

# 构建大模型
load_dotenv(override = True)

model = init_chat_model(
    model = "deepseek-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {"thinking": {"type": "disabled"}}
)

# 使用代码函数模拟代替外部API调用的天气查询
@tool(parse_docstring = True)
def get_weather(city: str = "北京"):
    """
    查询指定城市的天气

    Args:
        city : 城市名称
    """
    return f"{city}的天气阴雨连绵"

@tool(parse_docstring = True)
def get_news(domain: Literal["AI", "智能体"]):
    """
    查询特定领域的当日热点

    Args:
        domain: 特定领域
    """
    if domain == "AI":
        return "gpt新版本发布"
    elif domain == "智能体":
        return "Zcode发放免费token"
    else:
        return "666"


# 绑定工具到模型
tools = [get_news, get_weather]
model_with_tool = model.bind_tools(tools = tools)

res = model_with_tool.invoke(input = "查询上海的天气和智能体领域的新闻")
rprint(res)

In [13]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from random import randint
from langchain.messages import ToolMessage, SystemMessage


# 1.声明状态
class OverAllState(MessagesState):
    user_input: str
    final_output: str

# 2.声明结点
# 2.1 输入结点 将用户输入的查询信息 记录到message中 方便后续的大模型调用
def input_node(state: OverAllState) -> OverAllState:
    return {
        "messages": [HumanMessage(state["user_input"])]
    }


# 2.2 大模型结点
def llm_node(state: OverAllState) -> OverAllState:
    ai_msg = model_with_tool.invoke(state["messages"])
    return {
        "messages": ai_msg
    }

# 2.3 判断是否需要调用工具的结点
def tool_node(state: OverAllState) -> OverAllState:
    # 判断大模型的返回信息是否需要调用tool
    messages = state["messages"]
    ai_msg = messages[-1]
    tool_calls = ai_msg.tool_calls

    # 表示函数调用60%调用失败
    fail_prob = 6

    for tool_call in tool_calls:
        if tool_call["name"] == "get_weather":
            # 生成一个[0-9]数字判断大小
            if randint(0, 9) < fail_prob:
                messages.append(ToolMessage(
                    content = "网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                )
                )
            else:
                messages.append(get_weather.invoke(tool_call))
        elif tool_call["name"] == "get_news":
            if randint(0, 9) < fail_prob:
                messages.append(ToolMessage(
                    content = "网络波动，调用失败，请重试",
                    tool_call_id = tool_call["id"]
                )
                )
            else:
                messages.append(get_news.invoke(tool_call))
        else:
            messages.append(ToolMessage(
                content = "工具名称错误，调用失败请重试",
                tool_call_id = tool_call["id"]
            ))

    return {
        "messages": messages
    }

# 2.4 返回输出结点
def output_node(state: OverAllState) -> OverAllState:
    return {
        "final_output": state["messages"][-1].content
    }

# 2.5 判断是否需要工具结点的调用还是直接到output_node结点
def router(state: OverAllState) -> Literal["tool_node", "output_node"]:
    messages = state["messages"]
    last_msg = messages[-1]
    if last_msg.tool_calls:
        return "tool_node"
    return "output_node"

# 3.构建图
builder = StateGraph(state_schema = OverAllState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_conditional_edges("llm_node", router)
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()

ai_res = graph.invoke({
    "user_input": "查询今天的上海天气和智能体新闻热点",
    "messages": [SystemMessage("如果工具调用失败，必须重新调用直到成功为止")]
})

print("user_input:", ai_res["user_input"])
print("final_output:", ai_res["final_output"])

for msg in ai_res["messages"]:
    msg.pretty_print()

user_input: 查询今天的上海天气和智能体新闻热点
final_output: 两个查询都已成功完成，为您汇总如下：

## 🌧️ 上海天气
**阴雨连绵**，出门记得带伞哦 ☂️

## 📰 智能体领域热点
**Zcode 发放免费 token** —— 这是今日智能体领域的热点消息。

如需了解更多细节（比如上海具体温度、或这条新闻的详细内容），可以告诉我，我再帮您查询。
================================ System Message ================================

如果工具调用失败，必须重新调用直到成功为止
================================ Human Message =================================

查询今天的上海天气和智能体新闻热点
================================== Ai Message ==================================

我来为您查询上海天气和智能体领域的热点新闻。
Tool Calls:
  get_weather (call_00_yJ7i5pacB03ZgVzlk7tQ0640)
 Call ID: call_00_yJ7i5pacB03ZgVzlk7tQ0640
  Args:
    city: 上海
  get_news (call_01_NjbHZx5cpyWtoscJmgxs1005)
 Call ID: call_01_NjbHZx5cpyWtoscJmgxs1005
  Args:
    domain: 智能体
================================= Tool Message =================================

网络波动，调用失败，请重试
================================= Tool Message =================================

网络波动，调用失败，请重试
================================== Ai Message =====================